In [5]:
import pandas as pd
import numpy as np

segments = {
    'Champions':           {'Recency': 19.7,  'Frequency': 16.94, 'Monetary': 9208.34},
    'At-Risk':             {'Recency': 355.7, 'Frequency': 5.72,  'Monetary': 2532.54},
    'Big Spenders':        {'Recency': 316.5, 'Frequency': 1.52,  'Monetary': 2459.31},
    'Needs Attention':     {'Recency': 135.6, 'Frequency': 6.09,  'Monetary': 2268.49},
    'Potential Loyalists': {'Recency': 25.2,  'Frequency': 3.28,  'Monetary': 1355.11},
    'Loyal':               {'Recency': 22.7,  'Frequency': 5.47,  'Monetary': 935.93},
    'Promising':           {'Recency': 27.5,  'Frequency': 1.47,  'Monetary': 910.96},
    'General/Other':       {'Recency': 252.0, 'Frequency': 1.65,  'Monetary': 418.64},
    'Hibernating':         {'Recency': 556.2, 'Frequency': 1.20,  'Monetary': 297.30},
    'Lost':                {'Recency': 568.0, 'Frequency': 1.00,  'Monetary': 133.64},
}

all_recency   = [v['Recency']   for v in segments.values()]
all_frequency = [v['Frequency'] for v in segments.values()]
all_monetary  = [v['Monetary']  for v in segments.values()]

def normalize(value, all_values, invert=False):
    min_v = min(all_values)
    max_v = max(all_values)
    if max_v == min_v:
        return 0
    normalized = (value - min_v) / (max_v - min_v)
    # Add a floor of 0.1 so even the worst segment is visible
    normalized = 0.1 + normalized * 0.9
    return 1 - normalized if invert else normalized

# 3 axes at 90, 210, 330 degrees — top, bottom-left, bottom-right
angles_deg = [90, 210, 330]
scale = 5  # multiply all coordinates to spread the chart

dimensions = ['Recency', 'Frequency', 'Monetary']

rows = []
for seg, scores in segments.items():
    normalized_scores = {
        'Recency':   normalize(scores['Recency'],   all_recency,   invert=True),
        'Frequency': normalize(scores['Frequency'], all_frequency, invert=False),
        'Monetary':  normalize(scores['Monetary'],  all_monetary,  invert=False),
    }

    for i, dim in enumerate(dimensions):
        angle_rad = np.radians(angles_deg[i])
        score = normalized_scores[dim]
        rows.append({
            'segment':   seg,
            'dimension': dim,
            'raw_value': scores[dim],
            'score':     round(score, 4),
            'path':      i,
            'x':         round(score * scale * np.cos(angle_rad), 4),
            'y':         round(score * scale * np.sin(angle_rad), 4),
        })

    # Close the polygon
    angle_rad = np.radians(angles_deg[0])
    score = normalized_scores['Recency']
    rows.append({
        'segment':   seg,
        'dimension': 'Recency_close',
        'raw_value': scores['Recency'],
        'score':     round(score, 4),
        'path':      3,
        'x':         round(score * scale * np.cos(angle_rad), 4),
        'y':         round(score * scale * np.sin(angle_rad), 4),
    })

df = pd.DataFrame(rows)
df.to_csv('radar_data.csv', index=False)
print(df.sort_values(['segment', 'path']))

                segment      dimension  raw_value   score  path       x  \
4               At-Risk        Recency     355.70  0.3485     0  0.0000   
5               At-Risk      Frequency       5.72  0.3665     1 -1.5870   
6               At-Risk       Monetary    2532.54  0.3379     2  1.4632   
7               At-Risk  Recency_close     355.70  0.3485     3  0.0000   
8          Big Spenders        Recency     316.50  0.4128     0  0.0000   
9          Big Spenders      Frequency       1.52  0.1294     1 -0.5601   
10         Big Spenders       Monetary    2459.31  0.3307     2  1.4318   
11         Big Spenders  Recency_close     316.50  0.4128     3  0.0000   
0             Champions        Recency      19.70  0.9000     0  0.0000   
1             Champions      Frequency      16.94  1.0000     1 -4.3301   
2             Champions       Monetary    9208.34  1.0000     2  4.3301   
3             Champions  Recency_close      19.70  0.9000     3  0.0000   
28        General/Other  